**Audit an evaluation setup for leakage risks.**

Leakage inflates a score without ever raising an error: a duplicated sequence on both sides of a split, two windows of one protein pulled apart, a feature quietly derived from the label. The only symptom is an implausibly good number.

`aa.audit_leakage` runs a set of cheap heuristics over whatever parts of a setup are given and returns a plain table of findings, worst first, with the overall verdict in `df_audit.attrs["status"]`.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, GroupKFold, StratifiedKFold

import aaanalysis as aa

aa.options["verbose"] = False

# Twelve proteins, four windows each. The label is a property of the PROTEIN, so the
# windows of one protein are near-duplicates of each other.
rng = np.random.default_rng(0)
AA = list("ACDEFGHIKLMNPQRSTVWY")
n_prot, n_win = 12, 4

rows = []
for i in range(n_prot):
    parent = "".join(rng.choice(AA, 60))
    for w in range(n_win):
        rows.append({"entry": f"P{i}", "entry_win": f"P{i}_w{w}",
                     "sequence": parent, "window": parent[w:w + 12], "label": i % 2})
df_seq = pd.DataFrame(rows)
aa.display_df(df_seq, n_rows=10, show_shape=True)

DataFrame shape: (48, 5)


,entry,entry_win,sequence,window,label
1,P0,P0_w0,VPMGHACAETPWMPY...QMPGPSILYTYIQYQ,VPMGHACAETPW,0
2,P0,P0_w1,VPMGHACAETPWMPY...QMPGPSILYTYIQYQ,PMGHACAETPWM,0
3,P0,P0_w2,VPMGHACAETPWMPY...QMPGPSILYTYIQYQ,MGHACAETPWMP,0
4,P0,P0_w3,VPMGHACAETPWMPY...QMPGPSILYTYIQYQ,GHACAETPWMPY,0
5,P1,P1_w0,TQRIVDNRTMIHKLR...VCCQHNEVLVSRFSC,TQRIVDNRTMIH,1
6,P1,P1_w1,TQRIVDNRTMIHKLR...VCCQHNEVLVSRFSC,QRIVDNRTMIHK,1
7,P1,P1_w2,TQRIVDNRTMIHKLR...VCCQHNEVLVSRFSC,RIVDNRTMIHKL,1
8,P1,P1_w3,TQRIVDNRTMIHKLR...VCCQHNEVLVSRFSC,IVDNRTMIHKLR,1
9,P2,P2_w0,NKYEWCPNVGWQVES...MFSCKGRRRWWEDDR,NKYEWCPNVGWQ,0
10,P2,P2_w1,NKYEWCPNVGWQVES...MFSCKGRRRWWEDDR,KYEWCPNVGWQV,0


**`df_seq`: the dataset-level audit, before any split.**

With only `df_seq` the audit inspects the data itself. This is the check to run *before* choosing a split. Here nothing is wrong yet, so the table is empty and the status is `ok`.

In [2]:
df_audit = aa.audit_leakage(df_seq)

print("status:", df_audit.attrs["status"])
aa.display_df(df_audit, n_rows=10, show_shape=True)

status: ok
DataFrame shape: (0, 4)


,check,severity,detail,ids


Note that the windows share a parent `sequence` but each `window` is distinct. The audit compares `window` when it is present, so the repeated parent is not mistaken for a pile of duplicate samples. Duplicating an actual window does get flagged:

In [3]:
df_dup = df_seq.copy()
df_dup.loc[5, "window"] = df_dup.loc[0, "window"]   # P1_w1 is now a copy of P0_w0

df_audit = aa.audit_leakage(df_dup)
print("status:", df_audit.attrs["status"])
aa.display_df(df_audit, n_rows=10, show_shape=True)

status: medium
DataFrame shape: (1, 4)


,check,severity,detail,ids
1,duplicate_sequences,medium,2 rows share on...ready memorized,"['P0_w0', 'P1_w1']"


**`splits` and `labels`: the fold-level audit, after the split is built.**

`splits` takes the folds as integer row positions, exactly what a scikit-learn splitter yields, so `cv.split(...)` can be passed straight in. `labels` adds the class-balance check; it defaults to `df_seq['label']` when that column exists.

A plain `KFold` scatters the windows of each protein across the folds, which is the classic windowed-data leak.

In [4]:
labels = df_seq["label"].to_numpy()
splits_leaky = list(KFold(n_splits=4, shuffle=True, random_state=0).split(df_seq))

df_audit = aa.audit_leakage(df_seq, labels=labels, splits=splits_leaky)
print("status:", df_audit.attrs["status"])
aa.display_df(df_audit, n_rows=10, show_shape=True)

status: high
DataFrame shape: (1, 4)


,check,severity,detail,ids
1,same_protein_across_folds,high,12 protein(s) h...cross the split,"['P0', 'P1', 'P10', 'P11', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7']"


In [5]:
# The full sentence of the first finding, which is where the reasoning lives
for _, row in df_audit.iterrows():
    print(f"[{row['severity']}] {row['check']}\n    {row['detail']}\n    ids: {row['ids']}\n")

[high] same_protein_across_folds
    12 protein(s) have windows in both the training and the test part of fold(s) 0, 1, 2, 3, so neighbouring windows leak across the split
    ids: ['P0', 'P1', 'P10', 'P11', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7']



**`groups`: the same audit in the group vocabulary.**

`groups` is one label per sample, exactly as passed to `aa.bind_groups`: an accession, a family, or an externally computed homology cluster. Binding those groups to a group-aware splitter is the fix, and the audit confirms the leak is gone.

In [6]:
groups = df_seq["entry"].to_numpy()

cv = aa.bind_groups(GroupKFold(n_splits=4), groups=groups)
splits_clean = list(cv.split(df_seq))

df_audit = aa.audit_leakage(df_seq, labels=labels, groups=groups, splits=splits_clean)
print("status:", df_audit.attrs["status"])
aa.display_df(df_audit, n_rows=10, show_shape=True)

status: high
DataFrame shape: (1, 4)


,check,severity,detail,ids
1,class_balance_anomaly,high,the test part o...ially satisfied,"['0', '1', '2', '3']"


The protein and group leaks are gone. What remains is a *true* property of this setup: grouping by protein while the label is a property of the protein leaves some folds single-class. That is worth knowing before trusting a score, which is exactly the point.

**`X` and `names`: features that track the label too closely.**

`X` is the feature matrix and `names` gives one name per column, e.g. `df_feat['feature'].to_list()`, so a finding names the feature instead of its column position. A feature correlating with the label at `|r| >= 0.95` is what a target-derived column looks like.

In [7]:
X = rng.random((len(df_seq), 4))
X[:, 2] = labels + rng.random(len(df_seq)) * 0.01     # quietly derived from the target
names = ["hydrophobicity", "charge", "leaky_feature", "helix_propensity"]

df_audit = aa.audit_leakage(df_seq, labels=labels, X=X, names=names)
print("status:", df_audit.attrs["status"])
aa.display_df(df_audit, n_rows=10, show_shape=True)

status: high
DataFrame shape: (1, 4)


,check,severity,detail,ids
1,target_derived_feature,high,1 feature(s) co...thout the label,['leaky_feature']


**`raise_on`: turn a report into a hard failure.**

By default the audit only reports, so it can be dropped into a workflow without changing its control flow. `raise_on` sets the severity at or above which a finding raises a `ValueError` instead: it fires on a high-severity finding, and stays quiet when the worst finding sits below the chosen level.

In [8]:
try:
    aa.audit_leakage(df_seq, labels=labels, groups=groups, splits=splits_leaky,
                     raise_on="high")
except ValueError as e:
    print("ValueError:", str(e)[:300], "...")

ValueError: 'raise_on' ('high') matched 2 leakage finding(s) at or above that severity: same_protein_across_folds (high): 12 protein(s) have windows in both the training and the test part of fold(s) 0, 1, 2, 3, so neighbouring windows leak across the split; group_overlap_across_folds (high): 12 group(s) appear  ...


In [9]:
# Only a medium finding here, so raise_on="high" stays quiet and just returns the table
df_audit = aa.audit_leakage(df_dup, raise_on="high")
print("status:", df_audit.attrs["status"], "-> no exception raised")

# A clean setup never raises, whatever the threshold
df_clean = df_seq.drop(columns=["sequence"])
print("clean status:", aa.audit_leakage(df_clean, raise_on="low").attrs["status"])

status: medium -> no exception raised
clean status: ok


**The audit is heuristic, and a clean report is not a proof that no leakage exists.**

It inspects only the data and the folds it is given. It cannot see how a feature was built, whether a scaler was fitted before the split, or that two sequences are near-identical rather than byte-identical. Read `status='ok'` as *nothing obvious*, not as a guarantee. The severities are labels for a human reader, not a machine taxonomy: whether a finding must block a workflow is a decision this package leaves to you.